# Chope stress-test analysis

Run `chope_stress_test.py` first. This notebook reads its saved records; it does not launch a simulation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

EXPERIMENT_DIR = Path('../data/chope_experiments/REPLACE_WITH_EXPERIMENT_ID')
assert EXPERIMENT_DIR.exists(), f'Set EXPERIMENT_DIR to an existing result directory: {EXPERIMENT_DIR}'

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

runs = pd.DataFrame(read_jsonl(EXPERIMENT_DIR / 'runs.jsonl'))
convergence = pd.read_csv(EXPERIMENT_DIR / 'convergence.csv')
summary = json.loads((EXPERIMENT_DIR / 'summary.json').read_text(encoding='utf-8'))
runs[['run_id', 'model', 'status', 'outcome', 'resolution_achieved', 'persona_adherence_mean']]

In [ ]:
completed = runs.query("status == 'completed'").copy()
outcomes = (completed.groupby(['model', 'outcome']).size()
            .groupby(level=0).apply(lambda values: values / values.sum())
            .rename('proportion').reset_index())
sns.barplot(data=outcomes, x='outcome', y='proportion', hue='model')
plt.xticks(rotation=25)
plt.title('Outcome distribution by model');

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for metric, ax in zip(['outcome_tvd', 'max_binary_rate_change', 'max_continuous_smd', 'semantic_centroid_distance'], axes.flat):
    sns.lineplot(data=convergence, x='n', y=metric, hue='model', marker='o', ax=ax)
    ax.set_title(metric)
fig.suptitle('Cumulative convergence checks')
fig.tight_layout();

In [ ]:
resource_rows = []
for row in read_jsonl(EXPERIMENT_DIR / 'runs.jsonl'):
    for phase in ['simulation', 'evaluation', 'embedding', 'total']:
        resource_rows.append({'run_id': row['run_id'], 'model': row['model'], 'phase': phase, **row[f'{phase}_resources']})
resources = pd.DataFrame(resource_rows)
resources.groupby(['model', 'phase'])[['calls', 'total_tokens', 'cost_usd', 'local_latency_ms']].agg(['mean', 'sum'])

Before reporting a finding, open the linked transcripts in `manual_review.csv`, check the evaluator evidence, and inspect `boundary_events.jsonl`.